# Final Homework — Unsupervised Learning on NBA Player Data

**Course:** Introduction to Data Science
**Topic:** Dimensionality Reduction, Clustering, and Anomaly Detection

**Student name:** NOAM SHILO
**ID:** 322360975

---

This notebook explores the hidden structure of NBA player-season data using
unsupervised methods — with **no labels**. We reduce dimensions (PCA), look for
natural groups of players (clustering), and search for unusual players (anomaly
detection). The goal is not "nice" clusters, but understanding *why* each method
behaves as it does, when to trust it, and what it means for real players.


## ✏️ Edit map (read me first)

Everywhere you see the **✏️ EDIT** marker is a place to review and put into
your own words before submitting. They are:
- **2.1** correlation discussion
- **2.2** outlier exploration
- **3** PCA discussion (write the exact number of components from your run)
- **4.1** K-Means: the chosen `k` (code marker `# ✏️`)
- **4.2** DBSCAN: the `eps` parameter (code marker `# ✏️`)
- **4.4** feature-space clustering discussion
- **4.5** cluster evaluation + overall clustering discussion
- **5.2 / 5.3** the `contamination` and `n_neighbors` parameters (code markers `# ✏️`)
- **6** comparison discussion
- **7** critical analysis and ethics

The math-intuition text boxes (how each method works) are factual and can stay as
they are. The **✏️ EDIT** cells are the ones that reflect opinion/interpretation.

## 0. Setup and preprocessing

In [ ]:
# core libraries for data, plotting, stats and machine learning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from mpl_toolkits.mplot3d import Axes3D  # enables 3D plots

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from scipy.cluster.hierarchy import linkage, dendrogram

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
DATA_PATH = r"D:\USER\Desktop\data sience\all_seasons.csv.csv"

In [ ]:
# load the data and drop the unnamed index column if it exists
raw_df = pd.read_csv(DATA_PATH)
if raw_df.columns[0].lower().startswith("unnamed"):
    raw_df = raw_df.drop(columns=raw_df.columns[0])
df = raw_df.copy()
print("shape:", df.shape)
df.head()

### Choosing and scaling the features

Unsupervised methods here work on **numeric** features only, so we drop names,
teams and other text columns. We also **standardize** every feature (mean 0, std
1). This is essential: PCA, distance-based clustering and anomaly detection all
depend on distances, and without scaling a large-range feature like weight would
dominate a small-range feature like a shooting percentage. Standardizing puts
every feature on the same footing.

In [ ]:
# keep only the numeric features and remove rows with missing values
numeric_features = df.select_dtypes(include="number").columns.tolist()
data = df[numeric_features].dropna().copy()
print("features used:", numeric_features)
print("rows after dropna:", len(data))

# standardize every feature to mean 0 and std 1 so distances are fair
scaler = StandardScaler()
X = scaler.fit_transform(data)
X = pd.DataFrame(X, columns=numeric_features, index=data.index)
X.describe().round(2).loc[["mean", "std"]]

### Methodology overview

The pipeline is: select the 13 numeric features → drop missing rows → **standardize**
(so distances are fair) → explore (EDA) → reduce dimensions (PCA) → find groups
(3 clustering methods) → find unusual players (3 anomaly methods) → compare and
reflect. Standardization is the key preprocessing choice, because every method here
depends on distances and would otherwise be dominated by large-range features.

## 1. Dataset selection

**Where it comes from.** NBA player season-level statistics ("NBA Players",
`all_seasons.csv`), scraped from the official NBA stats API and published on
Kaggle.

**Why it was collected.** To track and compare player performance across seasons
— an operational record of the league, reused for analysis and journalism.

**Who collected it.** The NBA (official statistics), aggregated by the Kaggle
uploader.

**What each feature represents.**
- `age`, `player_height` (cm), `player_weight` (kg) — physical attributes
- `gp` — games played that season
- `pts`, `reb`, `ast` — points, rebounds, assists per game
- `net_rating` — team point differential per 100 possessions while the player is on court
- `oreb_pct`, `dreb_pct` — offensive / defensive rebound percentage
- `usg_pct` — usage rate (share of team plays the player uses)
- `ts_pct` — true shooting % (scoring efficiency)
- `ast_pct` — share of teammate baskets the player assisted

**Limitations / biases.** Rate stats depend on minutes and role, so comparing
across eras is unfair (the 3-point era inflates recent scoring). Survivorship
bias: weak players play few seasons and are under-represented. There is **no
labeled anomaly column** — which is exactly what this assignment requires.

**Requirements check:** 12,844 rows, 13 numeric features, real-world, no anomaly
label — all satisfied.


## 2. Exploratory Data Analysis

### 2.1 Structural analysis

In [ ]:
# rows, columns and data types
print(f"Rows: {df.shape[0]}   Columns: {df.shape[1]}")
print(df.dtypes)

In [ ]:
# missing values per column
missing = df.isna().sum()
print(missing[missing > 0] if missing.sum() else "No missing values in the columns used.")

In [ ]:
# statistical summary of the numeric features
data.describe().round(2).T

In [ ]:
# correlation matrix of the features - shows which features move together
plt.figure(figsize=(10, 8))
sns.heatmap(data.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix of numeric features")
plt.tight_layout()
plt.show()

> **✏️ EDIT — my draft; verify against your run and rewrite in your own words.**

**Discussion.** Several features are strongly related: `pts`, `usg_pct` and
`ast` move together (ball-dominant scorers), and `player_height`, `player_weight`,
`oreb_pct` and `dreb_pct` form a "big-man" group. This **redundancy** is exactly
what PCA will exploit — correlated features can be compressed into fewer
components with little information loss.

### 2.2 Distribution analysis

In [ ]:
# histogram + boxplot for each feature, side by side, to see shape and outliers
for col in numeric_features:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    sns.histplot(data[col], bins=30, kde=True, ax=axes[0])
    axes[0].set_title(f"{col} - distribution")
    sns.boxplot(x=data[col], ax=axes[1])
    axes[1].set_title(f"{col} - boxplot")
    plt.tight_layout()
    plt.show()

In [ ]:
# skewness and kurtosis measure the shape: skew = lean to one side, kurtosis = heavy tails
shape_table = pd.DataFrame({
    "skewness": data.skew(),
    "kurtosis": data.kurtosis(),
}).round(2).sort_values("kurtosis", ascending=False)
shape_table

> **✏️ EDIT — my draft; verify against your run and rewrite in your own words.**

**Outlier exploration.** Features like `pts`, `ast` and `ast_pct` are
right-skewed with high kurtosis — heavy tails of a few star players far above the
crowd. Some extreme values are **real** (a superstar season), while others hint at
**measurement artifacts**: players with very few games (`gp`) produce unstable
per-game rates, and rookies/undrafted players can show odd `net_rating`. These are
not data-entry errors, but they are noise that anomaly detection must handle
carefully — an extreme value is not automatically a true anomaly.

## 3. Dimensionality Reduction — PCA

**Mathematical intuition.** PCA finds new axes (principal components) that are
straight-line combinations of the original features, ordered so the first captures
the most variance, the second the next most (while being perpendicular to the
first), and so on. It rotates the data to point along the directions where it
spreads out the most, so a few components can describe most of the variation.

In [ ]:
# fit PCA on the standardized data and look at how much variance each component explains
pca = PCA()
scores = pca.fit_transform(X)

explained = pca.explained_variance_ratio_
cumulative = explained.cumsum()
for i, (e, c) in enumerate(zip(explained, cumulative), start=1):
    print(f"PC{i}: {e*100:5.1f}% | cumulative {c*100:5.1f}%")

In [ ]:
# scree plot - the explained variance per component, to see the "elbow"
plt.plot(range(1, len(explained)+1), explained*100, marker="o", label="individual")
plt.plot(range(1, len(cumulative)+1), cumulative*100, marker="s", label="cumulative")
plt.axhline(80, color="red", linestyle="--", label="80% line")
plt.title("Scree plot")
plt.xlabel("Principal component")
plt.ylabel("Explained variance (%)")
plt.legend()
plt.show()

In [ ]:
# project the data into 2D using the first two components
plt.figure(figsize=(8, 6))
plt.scatter(scores[:, 0], scores[:, 1], s=8, alpha=0.3)
plt.title("PCA - 2D projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
# project the data into 3D using the first three components
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(scores[:, 0], scores[:, 1], scores[:, 2], s=6, alpha=0.3)
ax.set_title("PCA - 3D projection")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
plt.show()

In [ ]:
# loadings - how much each original feature contributes to the first components
loadings = pd.DataFrame(pca.components_[:3].T, index=numeric_features,
                        columns=["PC1", "PC2", "PC3"]).round(2)
loadings

> **✏️ EDIT — my draft; verify against your run and rewrite in your own words.**

**Discussion.**
- **How many components?** The scree plot shows the cumulative curve crossing 80%
  after about 5-6 components, so ~5-6 of the 13 features are enough to represent
  the data well. (✏️ write the exact number your run shows.)
- **Information lost.** Reducing to 2D keeps only the variance in PC1+PC2 and drops
  the rest, so a 2D picture is a simplification.
- **Correlated variables & redundancy.** The strong correlations (scoring group,
  big-man group) mean the original 13 features are redundant; PCA packs that shared
  information into the first components.
- **Interpreting components.** From the loadings, PC1 is an overall "production /
  usage" axis (pts, usg_pct, ast), and PC2 separates big men (height, weight,
  rebounding) from guards. So the components carry real basketball meaning.
- **Local vs global structure, stability, cost.** PCA is a linear, global method:
  it captures global variance directions well and is stable and cheap to compute,
  but it can miss fine local structure that a nonlinear method would keep.
- **Better representation?** Yes — the reduced data removes redundancy and is easier
  to cluster and visualize, at the cost of some interpretability of the raw features.

## 4. Clustering Analysis

We apply three methods with different assumptions: **K-Means** (centroid-based),
**DBSCAN** (density-based), and **Hierarchical** (tree-based). We compare them on
the PCA projection.

### 4.1 K-Means

**Math intuition.** K-Means picks *k* centers and repeatedly (1) assigns each
point to its nearest center, (2) moves each center to the mean of its points,
until it stops changing. It minimizes the total squared distance of points to
their center. **Assumptions:** clusters are roughly round, similar-sized, and *k*
is known. **Strengths:** fast, simple. **Weaknesses:** must choose *k*, struggles
with non-round or different-density clusters, sensitive to outliers.

In [ ]:
# choose k with the elbow (inertia) and silhouette score
inertias, sils = [], []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=0, n_init=10).fit(X)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X, km.labels_, sample_size=3000, random_state=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o"); axes[0].set_title("Elbow (inertia)")
axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia")
axes[1].plot(list(k_range), sils, marker="o", color="#e8622c"); axes[1].set_title("Silhouette score")
axes[1].set_xlabel("k"); axes[1].set_ylabel("silhouette")
plt.tight_layout(); plt.show()

In [ ]:
# fit K-Means with the chosen k and show the clusters on the PCA projection
# ✏️ EDIT: k=4 was chosen from the elbow/silhouette above - change if you prefer another k
k_chosen = 4
kmeans = KMeans(n_clusters=k_chosen, random_state=0, n_init=10).fit(X)
labels_km = kmeans.labels_

plt.figure(figsize=(8, 6))
plt.scatter(scores[:, 0], scores[:, 1], c=labels_km, cmap="tab10", s=8, alpha=0.4)
plt.title(f"K-Means (k={k_chosen}) on PCA projection")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

In [ ]:
# describe each cluster by its average stats, to give it a real-world meaning
data_km = data.copy()
data_km["cluster"] = labels_km
cluster_profile = data_km.groupby("cluster")[["pts", "reb", "ast", "player_height", "usg_pct"]].mean().round(1)
print(cluster_profile)

### 4.2 DBSCAN

**Math intuition.** DBSCAN grows clusters from dense regions: a point with at
least `min_samples` neighbors within radius `eps` is a "core" point, and connected
core points form a cluster. Points in no dense region are labeled **noise (-1)**.
**Assumptions:** clusters are dense regions separated by sparse ones, of similar
density. **Strengths:** finds any shape, needs no *k*, marks outliers naturally.
**Weaknesses:** very sensitive to `eps`, struggles when clusters have different
densities, weak in high dimensions.

In [ ]:
# DBSCAN labels dense regions as clusters and the rest as noise (-1)
# ✏️ EDIT: eps and min_samples control DBSCAN - tune them for more/fewer clusters
dbscan = DBSCAN(eps=1.8, min_samples=15).fit(X)
labels_db = dbscan.labels_
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = (labels_db == -1).sum()
print(f"DBSCAN found {n_clusters_db} clusters and {n_noise} noise points")

plt.figure(figsize=(8, 6))
plt.scatter(scores[:, 0], scores[:, 1], c=labels_db, cmap="tab10", s=8, alpha=0.4)
plt.title("DBSCAN on PCA projection (noise = -1)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

### 4.3 Hierarchical clustering

**Math intuition.** Agglomerative hierarchical clustering starts with every
point as its own cluster and repeatedly merges the two closest clusters, building
a tree (dendrogram). Cutting the tree at a height gives the clusters.
**Assumptions:** a meaningful distance and linkage rule. **Strengths:** no *k*
needed up front, gives a full hierarchy. **Weaknesses:** O(n^2) memory, so it does
not scale — we run it on a sample.

In [ ]:
# hierarchical clustering is heavy, so run it on a random sample of 1500 rows
sample = X.sample(1500, random_state=0)
linkage_matrix = linkage(sample, method="ward")

plt.figure(figsize=(11, 4))
dendrogram(linkage_matrix, truncate_mode="lastp", p=30)
plt.title("Hierarchical clustering dendrogram (sample)")
plt.xlabel("samples"); plt.ylabel("distance")
plt.show()

hier = AgglomerativeClustering(n_clusters=4, linkage="ward").fit(sample)
plt.figure(figsize=(8, 6))
sample_scores = pca.transform(sample)
plt.scatter(sample_scores[:, 0], sample_scores[:, 1], c=hier.labels_, cmap="tab10", s=12, alpha=0.6)
plt.title("Hierarchical clusters on PCA projection (sample)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

### 4.4 Clustering the feature space

We also cluster the **features themselves** (the transpose of the data) to find
groups of related or redundant features.

In [ ]:
# cluster the features (columns) by how correlated they are, using a dendrogram on 1 - correlation
from scipy.spatial.distance import squareform
feature_dist = 1 - data.corr().abs()
feature_link = linkage(squareform(feature_dist.values, checks=False), method="average")
plt.figure(figsize=(11, 4))
dendrogram(feature_link, labels=numeric_features)
plt.title("Clustering of the feature space")
plt.ylabel("distance (1 - |correlation|)")
plt.tight_layout()
plt.show()

> **✏️ EDIT — my draft; verify against your run and rewrite in your own words.**

**Discussion.** Clustering the features groups together the redundant ones —
a "scoring/usage" group (pts, usg_pct, ts_pct) and a "big-man" group (height,
weight, rebound %). This confirms the redundancy PCA relies on, and shows which
features are near-duplicates versus which stand alone (e.g. `gp` or `net_rating`
sit apart). This kind of feature clustering is useful for feature selection and
for understanding dependencies in any high-dimensional dataset.

### 4.5 Cluster evaluation

In [ ]:
# evaluate with silhouette, and compare within-cluster variance to the global variance
sil_km = silhouette_score(X, labels_km, sample_size=3000, random_state=0)
print(f"K-Means silhouette: {sil_km:.3f}")

X_km = X.copy(); X_km["cluster"] = labels_km
global_var = X[numeric_features].var().mean()
within_var = X_km.groupby("cluster")[numeric_features].var().mean(axis=1).mean()
print(f"Global variance: {global_var:.3f} | mean within-cluster variance: {within_var:.3f}")

> **✏️ EDIT — my draft; verify against your run and rewrite in your own words.**

**Is within vs global variance a good metric?** Partly. Lower within-cluster
variance than the global variance means the clusters are tighter than the whole
data, which is a good sign. But this metric always improves as *k* grows, so on its
own it is misleading. A better companion is the **silhouette score**, which balances
how tight a cluster is against how far it is from the next cluster.

**Overall discussion.**
- **Why the methods disagree.** K-Means forces round, balanced groups; DBSCAN labels
  the dense center as one blob and pushes the star tail to noise; hierarchical gives
  a nested view. Each optimizes a different definition of "cluster".
- **Hyperparameter sensitivity.** K-Means depends on *k*, DBSCAN is very sensitive to
  `eps`/`min_samples`, hierarchical depends on the linkage rule.
- **Cluster geometry.** K-Means assumes round blobs; DBSCAN can follow any shape.
- **Density vs centroid.** Centroid methods (K-Means) split space by nearest center;
  density methods (DBSCAN) follow dense regions and isolate sparse points as noise.
- **When clustering fails.** When the data is a continuum with no real gaps — which is
  our case: player types blend smoothly, so there is no single "true" clustering.
  K-Means clusters still map nicely to player types (guards, wings, bigs, low-usage
  role players), which is the most interpretable result here.

## 5. Multi-Dimensional Anomaly Detection

This is the main part. We apply three methods and compare which players each one
flags as unusual.

### 5.1 Z-score method

**Idea.** For each feature, the z-score is how many standard deviations a value
is from the mean. We flag a row as anomalous if any feature's |z| exceeds 3.
**Assumption:** each feature is roughly Gaussian. **Why it can fail in high
dimensions:** it looks at one feature at a time, so it misses points that are
normal on every single feature but strange in *combination*; and with many
features, some feature crosses the threshold by chance for almost every row.

In [ ]:
# z-score: flag a row if any feature is more than 3 standard deviations from the mean
z_scores = np.abs(stats.zscore(data))
z_anom = (z_scores > 3).any(axis=1)
print(f"Z-score flagged {z_anom.sum()} rows ({z_anom.mean()*100:.1f}%) as anomalies")

### 5.2 Isolation Forest

**Isolation mechanism.** The algorithm builds many random trees: at each step it
picks a random feature and a random split. Anomalies are "few and different", so a
random split isolates them in **few steps** (short path), while normal points need
many splits. **Random partitioning** is what makes rare points separate quickly.
The **contamination** parameter sets the expected share of anomalies. The **anomaly
score** comes from the average path length — a **short path = high anomaly score**.
**Strengths:** works in high dimensions, no distance or Gaussian assumption, fast.
**Limitation:** results depend on the contamination guess.

In [ ]:
# Isolation Forest isolates rare points with short random-split paths
# ✏️ EDIT: contamination = expected share of anomalies (2%) - adjust to your judgment
iso = IsolationForest(contamination=0.02, random_state=0).fit(X)
iso_pred = iso.predict(X)          # -1 = anomaly, 1 = normal
iso_anom = iso_pred == -1
print(f"Isolation Forest flagged {iso_anom.sum()} anomalies")

plt.figure(figsize=(8, 6))
plt.scatter(scores[:, 0], scores[:, 1], c="lightgray", s=6, alpha=0.3, label="normal")
plt.scatter(scores[iso_anom, 0], scores[iso_anom, 1], c="red", s=14, label="anomaly")
plt.title("Isolation Forest anomalies on PCA projection")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.show()

### 5.3 Local Outlier Factor (LOF)

**Idea.** LOF compares the local density around a point to the density around
its `k` neighbors. A point in a much sparser area than its neighbors gets a high
LOF and is flagged. **Local density** is key: LOF finds points that are unusual
*relative to their neighborhood*, not just globally. **Sensitivity to k:** small
`k` reacts to tiny local gaps, large `k` behaves more globally. **Global vs local:**
LOF catches a point sitting in a low-density pocket even if it is not extreme
overall — something the global z-score misses.

In [ ]:
# LOF compares each point's local density to its neighbors' density
# ✏️ EDIT: n_neighbors=20 for LOF - try other values to test sensitivity to k
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
lof_pred = lof.fit_predict(X)      # -1 = anomaly
lof_anom = lof_pred == -1
print(f"LOF flagged {lof_anom.sum()} anomalies")

plt.figure(figsize=(8, 6))
plt.scatter(scores[:, 0], scores[:, 1], c="lightgray", s=6, alpha=0.3, label="normal")
plt.scatter(scores[lof_anom, 0], scores[lof_anom, 1], c="purple", s=14, label="anomaly")
plt.title("LOF anomalies on PCA projection")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.show()

## 6. Comparison of anomaly detection methods

In [ ]:
# compare which rows each method flagged and how much they agree
compare = pd.DataFrame({
    "zscore": z_anom,
    "iforest": iso_anom,
    "lof": lof_anom,
}, index=data.index)
compare["n_methods"] = compare.sum(axis=1)

print("Flagged by each method:")
print(compare[["zscore", "iforest", "lof"]].sum())
print("\nAgreement (how many methods flagged a row):")
print(compare["n_methods"].value_counts().sort_index())
print(f"\nFlagged by all three methods: {(compare['n_methods'] == 3).sum()}")

> **✏️ EDIT — my draft; verify against your run and rewrite in your own words.**

**Discussion.**
- **Agreement.** A small core of players is flagged by all three methods — the
  clearest anomalies (extreme superstar seasons or very unusual profiles).
- **Method-specific.** Z-score flags single-feature extremes; Isolation Forest and
  LOF flag multi-feature combinations. They disagree because each defines "unusual"
  differently (global-per-feature vs isolation vs local density).
- **Feature scaling.** All three depend on scaling; we standardized, without which
  large-range features would dominate.
- **Dimensionality.** Z-score degrades most as dimensions grow; Isolation Forest
  handles high dimensions best.
- **Runtime.** Z-score is instant, Isolation Forest is fast, LOF is the slowest (it
  needs neighbor searches).
- **Robustness to noise.** Isolation Forest and LOF tolerate noisy features better
  than the per-feature z-score, which reacts to every extreme value.
- **Interpretability.** Z-score is easiest to explain; the other two give scores that
  are harder to communicate.
- **False positives / negatives.** A false positive here is a legitimate star wrongly
  called an anomaly; a false negative is a genuinely odd record missed. In a real
  system (fraud, medical), a false negative can be far more costly than a false
  positive — the right trade-off depends on the application.

> **✏️ EDIT — my draft; verify against your run and rewrite in your own words.**

## 7. Critical Analysis and Reflection

**Why anomaly detection is ill-defined.** There is no label for "anomaly", so the
answer depends on the method, the parameters, and the definition of "unusual".
Different reasonable choices give different anomalies, and often more than one
answer is defensible.

**Curse of dimensionality.** As the number of features grows, points spread out and
the distance between the nearest and farthest neighbor becomes almost equal. So
distance-based ideas of "close" and "far" lose meaning, which weakens distance- and
density-based methods (LOF, and z-score's per-feature view).

**Noise vs genuine anomaly.** Noise is a random, meaningless deviation (a player
with 2 games and a wild per-game rate); a genuine anomaly is a real, meaningful
outlier (a historic superstar season). Example from our data: a player with very
few games can be flagged as anomalous purely because small samples give unstable
rates — that is noise, not a true anomaly.

**Risks of assuming Gaussian data.** Most of our features are skewed, not Gaussian.
The z-score assumes symmetry, so it over-flags the long right tail (many stars) and
can miss anomalies that hide inside a skewed shape.

**Why PCA plots can mislead.** A 2D PCA plot only shows the top two components and
drops the rest. Two points that look close in 2D may be far apart in the full
13-D space, so clusters or anomalies seen after reduction can be an artifact of the
projection rather than real structure.

### Ethical considerations
- **False alarms.** Too many false alarms make people ignore the system (alarm
  fatigue), so it becomes useless exactly when a real event happens.
- **Surveillance.** Flagging "abnormal" behavior can improve safety but risks
  privacy and can target people unfairly.
- **Medical.** Wrongly calling a patient anomalous causes needless worry and tests;
  missing a real anomaly can be dangerous — the cost of errors is not symmetric.
- **Cybersecurity.** Missing a real attack can be catastrophic, while false flags
  waste analysts' time — systems must balance the two.
- **Fairness.** Labeling individuals or groups as "abnormal" can encode bias, so
  anomaly systems must be checked for who they flag and why.
